In [ ]:
import os, json, glob, random, time, subprocess, sys
# Needs only the image's own torch + transformers. The NVARC utility script ships a Python-3.11 torch
# that shadows the image's torch on Python 3.13 -> strip it from every import path (kernel + workers).
sys.path[:] = [p for p in sys.path if "pip_install_unsloth" not in p]
os.environ["PYTHONPATH"] = ":".join(p for p in os.environ.get("PYTHONPATH", "").split(":") if p and "pip_install_unsloth" not in p)
print("python", sys.version.split()[0])
NOTEBOOK_START = time.time()
# ---- settings ---------------------------------------------------------------
MAX_MEGA     = 0        # 0 = all tasks in the uploaded sample; e.g. 400 for a quick test
N_PUBTRAIN   = 200      # public-training reference tasks (NVARC trained on these -> should score high)
DEADLINE_H   = 11.0     # workers stop here; results written so far are kept
# -----------------------------------------------------------------------------
COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
if not os.path.isdir(COMP):
    COMP = os.path.dirname(sorted(glob.glob("/kaggle/input/**/arc-agi_evaluation_challenges.json", recursive=True))[0])
_cands = sorted(glob.glob("/kaggle/input/**/qwen3_4b_grids15_sft139/transformers/bfloat16/1", recursive=True))
_cands = [c for c in _cands if os.path.exists(os.path.join(c, "config.json"))]
if not _cands:
    raise FileNotFoundError("NVARC model (qwen3_4b_grids15_sft139 bfloat16 v1) is not attached")
os.environ["NVARC_MODEL_DIR"] = _cands[0]
mega_ch = [p for p in glob.glob("/kaggle/input/**/challenges.json", recursive=True)
           if os.path.exists(os.path.join(os.path.dirname(p), "solutions.json"))]
if not mega_ch:
    raise FileNotFoundError("arc-mega-sample dataset (challenges.json + solutions.json) is not attached")
MEGA_DIR = os.path.dirname(mega_ch[0])
print("competition:", COMP); print("model:", _cands[0]); print("mega sample:", MEGA_DIR)

items = []
ch = json.load(open(f"{MEGA_DIR}/challenges.json")); so = json.load(open(f"{MEGA_DIR}/solutions.json"))
keys = sorted(ch); random.Random(0).shuffle(keys)
if MAX_MEGA: keys = keys[:MAX_MEGA]
for k in keys:
    t = ch[k]
    if not t["test"] or so.get(k) is None or not so[k] or so[k][0] is None: continue
    items.append({"key": k, "source": "mega", "train": t["train"], "test_input": t["test"][0]["input"], "test_output": so[k][0]})
ev = json.load(open(f"{COMP}/arc-agi_evaluation_challenges.json")); evs = json.load(open(f"{COMP}/arc-agi_evaluation_solutions.json"))
for k, t in ev.items():
    for i, x in enumerate(t["test"]):
        items.append({"key": f"{k}_{i}", "source": "eval", "train": t["train"], "test_input": x["input"], "test_output": evs[k][i]})
tr = json.load(open(f"{COMP}/arc-agi_training_challenges.json")); trs = json.load(open(f"{COMP}/arc-agi_training_solutions.json"))
for k in random.Random(1).sample(sorted(tr), N_PUBTRAIN):
    items.append({"key": f"{k}_0", "source": "pubtrain", "train": tr[k]["train"], "test_input": tr[k]["test"][0]["input"], "test_output": trs[k][0]})
json.dump(items, open("/kaggle/working/score_items.json", "w"))
from collections import Counter
print("items:", dict(Counter(i["source"] for i in items)))


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile mega_score_worker.py
"""Score NVARC's checkpoint (no test-time training) on a list of ARC test outputs.

One process per GPU. For each item: teacher-forced pass over (demos + test input + test output);
records the answer NLL and whether greedy decoding would reproduce the answer exactly
(= every answer token is the argmax given the correct prefix).

usage: python mega_score_worker.py RANK WORLD ITEMS_JSON OUT_JSONL END_TIME
"""
import json
import os
import sys
import time

rank, world = int(sys.argv[1]), int(sys.argv[2])
items_path, out_path, end_time = sys.argv[3], sys.argv[4], float(sys.argv[5])
os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
sys.path[:] = [p for p in sys.path if "pip_install_unsloth" not in p]   # never import the utility-script torch

import numpy as np  # noqa: E402
import torch  # noqa: E402
from transformers import AutoModelForCausalLM, AutoTokenizer  # noqa: E402

sys.path.insert(0, os.getcwd())
from arc_loader import QwenFormatter  # noqa: E402

MAXLEN = int(os.getenv("SCORE_MAXLEN", "8192"))
TOKEN_BUDGET = int(os.getenv("SCORE_TOKEN_BUDGET", "24576"))   # tokens per forward batch


def build(fmt, tok, it):
    """Token ids for prompt and answer; drop leading demos until it fits MAXLEN."""
    train = list(it["train"])
    while True:
        q = fmt.fmt_train(train) + fmt.fmt_query([{"input": it["test_input"]}])
        a = fmt.fmt_reply([it["test_output"]])
        qt, at = tok.encode(q), tok.encode(a)
        if len(qt) + len(at) <= MAXLEN:
            return qt, at, len(train)
        if len(train) <= 1:
            return None, None, 0
        train = train[1:]


def batches(prepped):
    """Group length-sorted items so that batch_size * max_len <= TOKEN_BUDGET."""
    prepped = sorted(prepped, key=lambda p: len(p[1]) + len(p[2]))
    cur, cur_max = [], 0
    for p in prepped:
        n = len(p[1]) + len(p[2])
        if cur and max(cur_max, n) * (len(cur) + 1) > TOKEN_BUDGET:
            yield cur
            cur, cur_max = [], 0
        cur.append(p)
        cur_max = max(cur_max, n)
    if cur:
        yield cur


@torch.no_grad()
def score_batch(model, batch, pad_id):
    seqs = [qt + at for _, qt, at, _ in batch]
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long, device="cuda")
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    hidden = model.model(input_ids=ids).last_hidden_state          # right padding + causal mask: real tokens unaffected
    out = []
    for i, (it, qt, at, ntr) in enumerate(batch):
        q = len(qt)
        h = hidden[i, q - 1:q - 1 + len(at)]
        logp = model.lm_head(h).float().log_softmax(-1)
        tgt = torch.tensor(at, device=logp.device)
        tok_lp = logp[torch.arange(len(at), device=logp.device), tgt]
        correct = (logp.argmax(-1) == tgt)
        out.append({"key": it["key"], "source": it["source"], "n_ans_tokens": len(at), "n_train_used": ntr,
                    "nll_sum": float(-tok_lp.sum()), "nll_mean": float(-tok_lp.mean()),
                    "greedy_exact": bool(correct.all()), "frac_tokens_correct": float(correct.float().mean()),
                    "seq_prob": float(torch.exp(tok_lp.sum()))})
    return out


def main():
    md = os.environ["NVARC_MODEL_DIR"]
    tok = AutoTokenizer.from_pretrained(md, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(md, torch_dtype=torch.bfloat16, local_files_only=True,
                                                 attn_implementation="sdpa").cuda().eval()
    pad_id = int(os.getenv("PAD_ID", "13"))
    fmt = QwenFormatter(tokenizer=tok)
    items = json.load(open(items_path))[rank::world]
    done = set()
    if os.path.exists(out_path):
        done = {json.loads(l)["key"] for l in open(out_path) if l.strip()}
    prepped, skipped = [], 0
    for it in items:
        if it["key"] in done:
            continue
        qt, at, ntr = build(fmt, tok, it)
        if qt is None:
            skipped += 1
            continue
        prepped.append((it, qt, at, ntr))
    print(f"[rank {rank}] {len(prepped)} to score, {len(done)} already done, {skipped} too long", flush=True)
    t0, n = time.time(), 0
    with open(out_path, "a") as f:
        for b in batches(prepped):
            if time.time() > end_time:
                print(f"[rank {rank}] deadline reached after {n} items", flush=True)
                break
            try:
                res = score_batch(model, b, pad_id)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                res = []
                for single in b:                      # retry one at a time
                    try:
                        res += score_batch(model, [single], pad_id)
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
            for r in res:
                f.write(json.dumps(r) + "\n")
            f.flush()
            n += len(b)
            if n % 500 < len(b):
                print(f"[rank {rank}] {n}/{len(prepped)} items, {n / (time.time() - t0):.2f} items/s", flush=True)
    print(f"[rank {rank}] finished {n} items in {(time.time() - t0) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# ---- run 4 workers (one per L4), heartbeat every 5 min ----------------------
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.version.cuda)
n_gpu = torch.cuda.device_count(); print("GPUs:", n_gpu)
if n_gpu == 0:
    raise RuntimeError("no GPU visible - set the accelerator to GPU L4x4")
end_time = NOTEBOOK_START + DEADLINE_H * 3600
procs = []
for r in range(n_gpu):
    log = open(f"/kaggle/working/worker{r}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "mega_score_worker.py", str(r), str(n_gpu), "/kaggle/working/score_items.json",
                                   f"/kaggle/working/scores_{r}.jsonl", str(end_time)], stdout=log, stderr=subprocess.STDOUT))
def _count():
    return sum(sum(1 for _ in open(p)) for p in glob.glob("/kaggle/working/scores_*.jsonl"))
while any(p.poll() is None for p in procs):
    time.sleep(300)
    tails = []
    for r in range(n_gpu):
        try: tails.append(open(f"/kaggle/working/worker{r}.log").read().strip().splitlines()[-1][-110:])
        except Exception: pass
    print(f"[{(time.time() - NOTEBOOK_START) / 3600:.2f} h] scored {_count()}/{len(items)} | " + " || ".join(tails), flush=True)
print("exit codes:", [p.returncode for p in procs])
for r in range(n_gpu):
    print(f"--- worker{r}.log (tail) ---"); print("\n".join(open(f"/kaggle/working/worker{r}.log").read().splitlines()[-8:]))


In [ ]:
# ---- summary ----------------------------------------------------------------
import csv, numpy as np
rows = [json.loads(l) for p in sorted(glob.glob("/kaggle/working/scores_*.jsonl")) for l in open(p) if l.strip()]
print(f"{len(rows)} scored")
for src in ("pubtrain", "eval", "mega"):
    r = [x for x in rows if x["source"] == src]
    if not r: continue
    g = np.mean([x["greedy_exact"] for x in r]); nm = np.median([x["nll_mean"] for x in r])
    p50 = np.mean([x["seq_prob"] > 0.5 for x in r]); p10 = np.mean([x["seq_prob"] < 0.1 for x in r])
    print(f"{src:<9} n={len(r):>6}  greedy-exact {100 * g:5.1f}%   seq_prob>0.5 {100 * p50:5.1f}%   seq_prob<0.1 {100 * p10:5.1f}%   median per-token NLL {nm:.4f}")
with open("/kaggle/working/mega_scores.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
hard = sorted([x for x in rows if x["source"] == "mega" and not x["greedy_exact"]], key=lambda x: -x["nll_mean"])
json.dump([x["key"] for x in hard], open("/kaggle/working/mega_hard_keys.json", "w"))
print(f"mega tasks NVARC gets wrong zero-shot: {len(hard)} -> mega_hard_keys.json; all scores -> mega_scores.csv")
print("Reading: pubtrain should be high (NVARC trained on it). If mega greedy-exact is close to pubtrain, Mega adds little;")
print("if it is far lower, there is headroom and mega_hard_keys.json is the candidate training set.")
